# Age of Empires with Gemma

**Experiment ID:** `exp-rts-001-thousand-turn-match`

This notebook starts a fresh Spartan civilization in **0 A.D. Release 28** and gives `google/gemma-4-E2B-it` control for up to 1,000 reinforcement-learning turns against PetraBot. The match starts from the standard scenario opening with a civic centre, starting units, starting resources, and no retained state from another run.

The final cells compare both players, select a conquest winner or turn-limit leader, and explain the result. The native replay recorder captures the complete command stream at maximum observer speed. FFmpeg then fits every captured frame into a video no longer than 45 seconds.

**Requirements:** a GPU runtime up to L4, access to the gated Gemma 4 repository on Hugging Face, and notebook access to the private Colab secrets `HF_WRITE_ACCESS`, `WANDB_KEY`, and `GITHUB_TOKEN`. The first run downloads the official 0 A.D. AppImage and Gemma weights.

In [ ]:
# @title 1. Install the Gemma runtime
%pip install -q -U "transformers>=5.10.1" accelerate bitsandbytes huggingface_hub wandb
%pip install -q "git+https://github.com/ritwikraha/markov-chainsaw.git@age-of-llms#subdirectory=age-of-llms"

In [ ]:
# @title 2. Imports, reproducibility, and private Hugging Face login
import atexit, collections, hashlib, json, math, os, pathlib, random, re
import shutil, signal, subprocess, sys, time, urllib.request, zipfile
from dataclasses import dataclass
from typing import Any

from IPython.display import Markdown, Video, display
import pandas as pd
import torch
from huggingface_hub import login
import wandb
from utils import capture_complete_replay, compare_players, file_sha256, make_replay_archive, replay_metadata

EXPERIMENT_ID = "exp-rts-001-thousand-turn-match"
SEED = 1001
random.seed(SEED)
torch.manual_seed(SEED)
SMOKE_TEST = os.environ.get("COLAB_SMOKE_TEST") == "1"

def required_secret(name: str) -> str:
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        raise RuntimeError(f"Missing {name}. Add it under Colab > Secrets and grant notebook access.")
    return value

if not SMOKE_TEST:
    hf_token = required_secret("HF_WRITE_ACCESS")
    wandb_key = required_secret("WANDB_KEY")
    github_token = required_secret("GITHUB_TOKEN")
    login(token=hf_token, add_to_git_credential=False)
    wandb.login(key=wandb_key, relogin=True)
    github_request = urllib.request.Request(
        "https://api.github.com/user",
        headers={"Authorization": f"Bearer {github_token}", "Accept": "application/vnd.github+json"},
    )
    with urllib.request.urlopen(github_request, timeout=20) as response:
        if response.status != 200: raise RuntimeError("GitHub authentication failed.")
    WANDB_RUN = wandb.init(
        project="gemma4-zero-ad", job_type="thousand-turn-experiment",
        name=EXPERIMENT_ID,
        config={"experiment_id": EXPERIMENT_ID, "model": "google/gemma-4-E2B-it", "game": "0 A.D. 0.28.0", "seed": SEED, "turn_limit": 1000},
    )
    atexit.register(wandb.finish)
    del hf_token, wandb_key, github_token
    print("Hugging Face, Weights & Biases, and GitHub authentication succeeded.")
else:
    WANDB_RUN = None
    print("CLI smoke-test mode: external authentication and Gemma download are intentionally skipped.")

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > GPU, then run again.")
capability = torch.cuda.get_device_capability(0)
COMPUTE_DTYPE = torch.bfloat16 if capability[0] >= 8 else torch.float16
print(f"Ready on {torch.cuda.get_device_name(0)}; compute dtype: {COMPUTE_DTYPE}.")

## Install 0 A.D. Release 28

The AppImage and Python client below are official Wildfire Games artifacts pinned to Release 28. Downloads are SHA-256 verified. Pyrogenesis refuses to run as root, so the notebook creates a dedicated unprivileged local account for the engine. Its RL service listens only on `127.0.0.1`.

In [ ]:
# @title 3. Download and verify the game and matching zero_ad client
APPIMAGE_URL = "https://releases.wildfiregames.com/0ad-0.28.0-x86_64.AppImage"
APPIMAGE_SHA256 = "869dd85adcb4e02b7dc8f76853a20003b1c50e328fc43b3a12336f33570add18"
CLIENT_BASE = "https://gitea.wildfiregames.com/0ad/0ad/raw/tag/v0.28.0/source/tools/rlclient/python"
CLIENT_FILES = {
    "zero_ad/__init__.py": "5b19a6a28852cff228bb9d920162e00046604db6f9c99611031f96a4cc377c3b",
    "zero_ad/actions.py": "3da601d8fe674b45312ac27315dd12dc241e56382770f46aa0f2cced1379289c",
    "zero_ad/api.py": "a276f59bfb1fc8d043dd6e37e654cbb9b8943b6293f69f2e626143f9f74609dc",
    "zero_ad/environment.py": "e42ee426779ac8f519648ed56356d01cb3bd8c12fa07bf1dacd1c880ecbceeaa",
    "samples/arcadia.json": "c647bb6c2c3bf672695d248c4822b32e8dc6a4d5f04a63be6be5a2f7ec95ee12",
}

def sha256(path: pathlib.Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def download_verified(url: str, destination: pathlib.Path, expected: str) -> pathlib.Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if not destination.exists() or sha256(destination) != expected:
        print(f"Downloading {destination.name}...")
        urllib.request.urlretrieve(url, destination)
    actual = sha256(destination)
    if actual != expected:
        destination.unlink(missing_ok=True)
        raise RuntimeError(f"Checksum mismatch for {destination.name}: {actual}")
    return destination

ENGINE_ROOT = pathlib.Path("/content/zero-ad-0.28.0")
EXTRACTED_ROOT = ENGINE_ROOT / "squashfs-root"
APPIMAGE_PATH = ENGINE_ROOT / "0ad-0.28.0-x86_64.AppImage"
APP_RUN = EXTRACTED_ROOT / "AppRun"
if not APP_RUN.exists():
    download_verified(APPIMAGE_URL, APPIMAGE_PATH, APPIMAGE_SHA256)
    APPIMAGE_PATH.chmod(0o755)
    subprocess.run([str(APPIMAGE_PATH), "--appimage-extract"], cwd=ENGINE_ROOT, check=True, stdout=subprocess.DEVNULL)
    APPIMAGE_PATH.unlink()  # extracted copy is enough; reclaim ~1.7 GB
if not APP_RUN.exists():
    raise FileNotFoundError(f"0 A.D. extraction did not produce {APP_RUN}")

CLIENT_ROOT = pathlib.Path("/content/zero-ad-client-r28")
for relative, expected in CLIENT_FILES.items():
    download_verified(f"{CLIENT_BASE}/{relative}", CLIENT_ROOT / relative, expected)
sys.path.insert(0, str(CLIENT_ROOT))
import zero_ad

RUNNER = "zero_ad_runner"
if subprocess.run(["id", "-u", RUNNER], capture_output=True).returncode:
    subprocess.run(["useradd", "-m", "-s", "/bin/bash", RUNNER], check=True)
version = subprocess.run(
    ["runuser", "-u", RUNNER, "--", "env", f"HOME=/home/{RUNNER}", str(APP_RUN), "-version"],
    check=True, capture_output=True, text=True,
)
assert "0.28.0" in version.stdout + version.stderr
print((version.stdout + version.stderr).strip().splitlines()[-1], "and zero_ad client are ready.")

In [ ]:
# @title 4. Launch the real engine and reset a Spartan-vs-Petra match
class ZeroADEngine:
    def __init__(self, executable: pathlib.Path, runner: str):
        self.executable, self.runner, self.process, self.log = executable, runner, None, None

    def start(self):
        self.stop()
        log_path = pathlib.Path("/content/zero_ad_engine.log")
        self.log = log_path.open("w")
        command = [
            "runuser", "-u", self.runner, "--", "env", f"HOME=/home/{self.runner}",
            str(self.executable), "-autostart-nonvisual", "-autostart=random/alpine_lakes",
            "-autostart-players=2", "-autostart-player=-1",
            "-autostart-ai=1:petra", "-autostart-ai=2:petra",
            "-rl-interface=127.0.0.1:6000", "-quickstart", "-nosound",
        ]
        self.process = subprocess.Popen(command, stdout=self.log, stderr=subprocess.STDOUT)
        return self

    def stop(self):
        if self.process and self.process.poll() is None:
            self.process.send_signal(signal.SIGTERM)
            try:
                self.process.wait(timeout=20)
            except subprocess.TimeoutExpired:
                self.process.kill(); self.process.wait()
        if self.log and not self.log.closed:
            self.log.close()

previous_engine = globals().get("engine")
if previous_engine is not None and hasattr(previous_engine, "stop"):
    previous_engine.stop()
engine = ZeroADEngine(APP_RUN, RUNNER).start()
atexit.register(engine.stop)
game = zero_ad.ZeroAD("http://127.0.0.1:6000")

scenario = json.loads((CLIENT_ROOT / "samples/arcadia.json").read_text())
scenario["settings"]["Name"] = f"{EXPERIMENT_ID}: Gemma 4 vs Petra"
scenario["settings"]["Seed"] = SEED
scenario["settings"]["AISeed"] = SEED
scenario["settings"]["PlayerData"][0]["AI"] = ""
scenario["settings"]["PlayerData"][1]["AI"] = "petra"
scenario["settings"]["PlayerData"][1]["AIDiff"] = 3
scenario["gameSpeed"] = 15

last_error = None
for _ in range(90):
    if engine.process.poll() is not None:
        raise RuntimeError("0 A.D. exited during startup; inspect /content/zero_ad_engine.log")
    try:
        state = game.reset(json.dumps(scenario), save_replay=True, player_id=1)
        break
    except Exception as exc:
        last_error = exc; time.sleep(1)
else:
    raise RuntimeError(f"0 A.D. RL interface was not ready: {last_error}")
initial_state = observation_data = {
    "gemma_entities": len(state.units(owner=1)),
    "petra_entities": len(state.units(owner=2)),
    "gemma_resources": state.data["players"][1].get("resourceCounts", {}),
    "petra_resources": state.data["players"][2].get("resourceCounts", {}),
}
print("Fresh standard opening loaded:", json.dumps(initial_state, indent=2))

In [ ]:
# @title 5. Compact observations, legal actions, and engine adapter
ALLOWED_ACTIONS = {
    "gather_food", "gather_wood", "gather_stone", "gather_metal",
    "train_civilian", "train_spearman", "build_house", "build_barracks",
    "scout", "attack_enemy_unit", "attack_enemy_structure", "defend", "wait",
}

@dataclass(frozen=True)
class CommanderAction:
    action: str
    count: int = 1
    reason: str = ""

def extract_first_json_object(text: str) -> dict[str, Any]:
    decoder = json.JSONDecoder()
    for index, character in enumerate(text):
        if character == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
                if isinstance(value, dict): return value
            except json.JSONDecodeError:
                pass
    raise ValueError("No JSON object found.")

def parse_action(text: str, legal: list[str]) -> CommanderAction:
    payload = extract_first_json_object(text)
    action = str(payload.get("action", "")).strip().lower()
    if action not in ALLOWED_ACTIONS or action not in legal:
        raise ValueError(f"Illegal action: {action!r}")
    try:
        count = max(1, min(int(payload.get("count", 1)), 8))
    except (TypeError, ValueError) as exc:
        raise ValueError("count must be an integer") from exc
    return CommanderAction(action, count, str(payload.get("reason", ""))[:160])

def positioned(units):
    return [unit for unit in units if "position" in unit.data]

def centre(units):
    points = [unit.position() for unit in positioned(units)]
    return [sum(point[i] for point in points) / len(points) for i in (0, 1)] if points else [0, 0]

def closest(units, position):
    units = positioned(units)
    return min(units, key=lambda unit: sum((a - b) ** 2 for a, b in zip(unit.position(), position))) if units else None

def workers(state):
    return positioned([u for u in state.units(owner=1) if u.type().startswith("units/") and any(x in u.type() for x in ("civilian", "infantry"))])

def military(state, owner=1):
    return positioned([u for u in state.units(owner=owner) if u.type().startswith("units/") and "support_civilian" not in u.type()])

def resource_nodes(state, resource):
    needles = {"food": ("fruit/", "fauna_", "fauna/", "fish"), "wood": ("tree/",), "stone": ("rock/",), "metal": ("ore/", "metal")}[resource]
    return positioned([u for u in state.units(owner=0) if any(needle in u.type() for needle in needles)])

def observation(state):
    player, enemy = state.data["players"][1], state.data["players"][2]
    friendly_structures = [u for u in state.units(owner=1) if u.type().startswith("structures/")]
    enemy_structures = [u for u in state.units(owner=2) if u.type().startswith("structures/")]
    return {
        "minute": round(state.data.get("timeElapsed", 0) / 60_000, 1),
        "phase": player.get("phase"), "resources": player.get("resourceCounts", {}),
        "population": {"used": player.get("popCount"), "limit": player.get("popLimit")},
        "economy": {"workers": len(workers(state)), "resource_gatherers": player.get("resourceGatherers", {})},
        "army": len(military(state)), "structures": collections.Counter(u.type().split("/")[-1] for u in friendly_structures),
        "enemy": {"state": enemy.get("state"), "units": len(military(state, 2)), "structures": len(enemy_structures)},
        "own_state": player.get("state"), "objective": "Defeat PetraBot by building an economy and destroying its units.",
    }

def legal_actions(state):
    legal = ["wait"]
    player = state.data["players"][1]
    resources = player.get("resourceCounts", {})
    population_room = player.get("popLimit", 0) - player.get("popCount", 0)
    own_workers = workers(state)
    for resource in ("food", "wood", "stone", "metal"):
        if own_workers and resource_nodes(state, resource): legal.append(f"gather_{resource}")
    if state.units(owner=1, entity_type="civil_centre") and population_room > 0:
        if resources.get("food", 0) >= 50: legal.append("train_civilian")
        if resources.get("food", 0) >= 50 and resources.get("wood", 0) >= 50: legal.append("train_spearman")
    if own_workers and resources.get("wood", 0) >= 100:
        legal.append("build_house")
    if own_workers and resources.get("wood", 0) >= 200 and not state.units(owner=1, entity_type="barracks"):
        legal.append("build_barracks")
    own_army, enemy_units = military(state), positioned(state.units(owner=2))
    if own_army and enemy_units:
        legal.extend(["scout", "attack_enemy_unit", "defend"])
        if any(u.type().startswith("structures/") for u in enemy_units): legal.append("attack_enemy_structure")
    return legal

BUILD_NUMBER = 0
def engine_commands(state, command: CommanderAction):
    global BUILD_NUMBER
    own_workers = workers(state)
    civic_centres = positioned(state.units(owner=1, entity_type="civil_centre"))
    if command.action.startswith("gather_"):
        resource = command.action.removeprefix("gather_")
        selected = own_workers[:command.count]
        target = closest(resource_nodes(state, resource), centre(selected))
        return [zero_ad.actions.gather(selected, target)] if selected and target else []
    player = state.data["players"][1]
    resources = player.get("resourceCounts", {})
    population_room = player.get("popLimit", 0) - player.get("popCount", 0)
    if command.action == "train_civilian" and civic_centres:
        count = min(command.count, resources.get("food", 0) // 50, population_room)
        return [zero_ad.actions.train(civic_centres, "units/spart/support_civilian", count)] if count else []
    if command.action == "train_spearman":
        producers = positioned(state.units(owner=1, entity_type="barracks")) or civic_centres
        count = min(command.count, resources.get("food", 0) // 50, resources.get("wood", 0) // 50, population_room)
        return [zero_ad.actions.train(producers, "units/spart/infantry_spearman_b", count)] if producers and count else []
    if command.action.startswith("build_") and own_workers and civic_centres:
        building = command.action.removeprefix("build_")
        template = {"house": "structures/spart/house", "barracks": "structures/spart/barracks"}[building]
        angle = BUILD_NUMBER * math.pi / 4; radius = 30 + 12 * (BUILD_NUMBER // 8)
        x, z = civic_centres[0].position(); BUILD_NUMBER += 1
        return [zero_ad.actions.construct(own_workers[:3], template, x + radius * math.cos(angle), z + radius * math.sin(angle))]
    own_army = military(state)
    enemies = positioned(state.units(owner=2))
    if command.action == "scout" and own_army and enemies:
        return [zero_ad.actions.walk(own_army[:command.count], *centre(enemies))]
    if command.action in ("attack_enemy_unit", "attack_enemy_structure") and own_army:
        prefix = "units/" if command.action.endswith("unit") else "structures/"
        targets = [u for u in enemies if u.type().startswith(prefix)]
        target = closest(targets, centre(own_army))
        return [zero_ad.actions.attack(own_army, target)] if target else []
    if command.action == "defend" and own_army and civic_centres:
        target = closest(enemies, civic_centres[0].position())
        return [zero_ad.actions.attack(own_army, target)] if target else []
    return []

assert parse_action('{"action":"gather_wood","count":3}', ["gather_wood"]).count == 3
print(json.dumps(observation(state), indent=2, default=dict))

In [ ]:
# @title 6. Deterministic fallback policy
def heuristic_action(state) -> CommanderAction:
    obs, legal = observation(state), legal_actions(state)
    resources, population = obs["resources"], obs["population"]
    structures = obs["structures"]
    priority = []
    if population["limit"] - population["used"] <= 3: priority.append("build_house")
    if obs["economy"]["workers"] < 12: priority.append("train_civilian")
    if resources.get("food", 0) < 250: priority.append("gather_food")
    if resources.get("wood", 0) < 250: priority.append("gather_wood")
    if not structures.get("barracks"): priority.append("build_barracks")
    if obs["army"] < 10: priority.append("train_spearman")
    priority.extend(["attack_enemy_unit", "defend", "gather_wood", "wait"])
    choice = next((candidate for candidate in priority if candidate in legal), legal[0])
    return CommanderAction(choice, 2 if choice.startswith(("train_", "gather_")) else 1, "deterministic fallback")

print("Fallback opening:", heuristic_action(state))

## Load Gemma 4

E2B is the practical Gemma 4 default for a T4. Four-bit loading leaves memory for 0 A.D.; choose a larger checkpoint only on a larger GPU.

In [ ]:
# @title 7. Load Gemma 4 in 4-bit mode
MODEL_ID = "google/gemma-4-E2B-it" # @param {type:"string"}
if not SMOKE_TEST:
    from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization, device_map="auto", dtype=COMPUTE_DTYPE
    )
    model.eval()
    print(f"Loaded {MODEL_ID}.")
else:
    model = processor = None

In [ ]:
# @title 8. Gemma strategic commander
SYSTEM_PROMPT = """You command Sparta in a real single-player 0 A.D. match against PetraBot.
Build a sustainable food/wood economy, avoid population blocks, train civilians and spearmen, construct houses and a barracks, defend threats, and attack when strong.
Return exactly one compact JSON object and nothing else:
{"action": "one legal action", "count": 1, "reason": "brief tactical reason"}
Choose an action from the supplied list. count must be 1-8. The legal-action list contains affordable choices. If a resource is low, gather it. Change strategy when population or resources remain unchanged.
"""

class GemmaCommander:
    def __init__(self, model, processor): self.model, self.processor = model, processor

    @torch.inference_mode()
    def choose(self, obs, legal):
        prompt = "LIVE 0 A.D. STATE:\n" + json.dumps(obs, sort_keys=True, default=dict) + "\nLEGAL ACTIONS:\n" + json.dumps(legal) + "\nChoose one move. JSON only."
        messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [{"type": "text", "text": prompt}]},
        ]
        inputs = self.processor.apply_chat_template(
            messages, tokenize=True, return_dict=True, return_tensors="pt",
            add_generation_prompt=True, enable_thinking=False,
        ).to(self.model.device)
        prompt_length = inputs["input_ids"].shape[-1]
        generated = self.model.generate(
            **inputs, max_new_tokens=96, do_sample=False,
            pad_token_id=self.processor.tokenizer.eos_token_id,
        )
        raw = self.processor.decode(generated[0][prompt_length:], skip_special_tokens=True).strip()
        return parse_action(raw, legal), raw

commander = None if SMOKE_TEST else GemmaCommander(model, processor)

In [ ]:
# @title 9. Run the fresh civilization for up to 1,000 turns
MAX_TURNS = 1000 # @param {type:"integer"}
DECISION_INTERVAL = 25 # @param {type:"integer"}
MAX_MODEL_FAILURES = 5 # @param {type:"integer"}
if SMOKE_TEST:
    MAX_TURNS, DECISION_INTERVAL = 12, 4

def tactical_snapshot(state, turn, decision, action):
    return {
        "turn": turn, "decision": decision, "action": action.action,
        "reason": action.reason, "observation": observation(state),
    }

trajectory, failures, completed_turns, decision = [], 0, 0, 0
active_commands = []
try:
    for turn in range(1, MAX_TURNS + 1):
        obs = observation(state)
        if obs["own_state"] != "active" or obs["enemy"]["state"] != "active":
            break
        if turn == 1 or (turn - 1) % DECISION_INTERVAL == 0:
            decision += 1
            legal = legal_actions(state)
            try:
                if SMOKE_TEST:
                    action, raw = heuristic_action(state), "CLI smoke-test policy"
                else:
                    action, raw = commander.choose(obs, legal)
            except Exception as exc:
                failures += 1
                if failures > MAX_MODEL_FAILURES: raise
                action, raw = heuristic_action(state), f"fallback after {type(exc).__name__}: {exc}"
            active_commands = engine_commands(state, action)
        else:
            active_commands = []
        state = game.step(active_commands)
        completed_turns = turn
        if turn == 1 or (turn - 1) % DECISION_INTERVAL == 0 or turn == MAX_TURNS:
            trajectory.append(tactical_snapshot(state, turn, decision, action))
            now = observation(state)
            print(f"T{turn:04d} D{decision:02d} {now['minute']:>5.1f}m | {action.action:<24} | pop {now['population']['used']}/{now['population']['limit']} | army {now['army']} | enemy {now['enemy']['units']}")
            if WANDB_RUN is not None:
                WANDB_RUN.log({
                    "game/turn": turn, "game/minute": now["minute"],
                    "game/population": now["population"]["used"], "game/army": now["army"],
                    "game/enemy_units": now["enemy"]["units"],
                    "game/food": now["resources"].get("food", 0), "decision/action": action.action,
                }, step=turn)
finally:
    engine.stop()

final = observation(state)
outcome = compare_players(state)
print(f"\nCompleted {completed_turns} RL turns and {decision} Gemma decisions.")
print(f"Result: {outcome.winner} ({outcome.result_type}). {outcome.reason}")
print(f"Parser fallbacks: {failures}.")

In [ ]:
# @title 10. Capture the complete native replay and fit it within 45 seconds
MAXIMUM_VIDEO_SECONDS = 45 # @param {type:"integer"}
RENDER_SAFETY_FACTOR = 20.0 # @param {type:"number"}
if SMOKE_TEST:
    MAXIMUM_VIDEO_SECONDS, RENDER_SAFETY_FACTOR = 10, 1.0

replay_root = pathlib.Path(f"/home/{RUNNER}/.local/share/0ad/replays")
commands_files = list(replay_root.rglob("commands.txt"))
if not commands_files:
    raise FileNotFoundError(f"No replay found under {replay_root}")
replay_commands = max(commands_files, key=lambda path: path.stat().st_mtime)
latest_replay = replay_commands.parent
VIDEO_PATH = pathlib.Path(f"/content/{EXPERIMENT_ID}.mp4")

subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "xvfb", "x11-utils", "xdotool", "ffmpeg", "mesa-utils"],
    check=True, stdout=subprocess.DEVNULL,
)
metadata = replay_metadata(replay_commands)
print(f"Native replay contains {metadata['final_turn']} simulation turns and {metadata['simulation_seconds']:.1f} simulated seconds.")
capture = capture_complete_replay(
    APP_RUN, replay_commands, VIDEO_PATH, runner=RUNNER,
    maximum_final_seconds=MAXIMUM_VIDEO_SECONDS,
    renderer_safety_factor=RENDER_SAFETY_FACTOR,
    maximum_capture_seconds=1200,
)
print(f"Complete capture: {capture.raw_seconds:.1f}s. Final video: {capture.final_seconds:.1f}s.")
print(f"Video SHA-256: {file_sha256(VIDEO_PATH)}")
display(Video(str(VIDEO_PATH), embed=True))

In [ ]:
# @title 11. Compare final Gemma and Petra statistics
rows = []
for frame in trajectory:
    obs = frame["observation"]
    rows.append({
        "turn": frame["turn"], "decision": frame["decision"], "minute": obs["minute"], "action": frame["action"],
        "reason": frame["reason"], "population": obs["population"]["used"],
        "army": obs["army"], "enemy_units": obs["enemy"]["units"], **obs["resources"],
    })
decision_table = pd.DataFrame(rows)
display(decision_table)
display(pd.DataFrame(outcome.rows()).set_index("Metric"))
display(Markdown(outcome.markdown()))
display(Markdown(f"**Winner: {outcome.winner}**  \nResult type: {outcome.result_type}  \n{outcome.reason}"))

In [ ]:
# @title 12. Package and download the experiment artifacts
archive_base = pathlib.Path(f"/content/{EXPERIMENT_ID}-release28-replay")
archive_path = make_replay_archive(latest_replay, archive_base)
print(f"Created {archive_path} ({archive_path.stat().st_size / 1_000_000:.1f} MB).")
print(f"Replay SHA-256: {file_sha256(archive_path)}")
print("To watch with the native 3D renderer, unzip this folder into your 0 A.D. Release 28 replay directory, then open the Replays screen from the main menu.")
if WANDB_RUN is not None:
    WANDB_RUN.summary.update({"winner": outcome.winner, "result_type": outcome.result_type, "completed_turns": completed_turns})
    artifact = wandb.Artifact(EXPERIMENT_ID, type="gameplay")
    artifact.add_file(str(VIDEO_PATH)); artifact.add_file(str(archive_path))
    WANDB_RUN.log_artifact(artifact)
    wandb.finish()
if SMOKE_TEST:
    print(VIDEO_PATH, archive_path)
else:
    try:
        from google.colab import files
        files.download(str(VIDEO_PATH)); files.download(str(archive_path))
    except ImportError:
        print(archive_path)

## What is being automated

Gemma controls a local, offline match through 0 A.D.'s supported developer/RL interface. Model output cannot invoke arbitrary engine commands: it must pass the action allowlist, legal-action check, bounded count, and macro adapter. The engine endpoint binds to localhost and the process is terminated after the match.

The MP4 is captured directly from 0 A.D.'s `-replay-visual` mode and uses the actual game renderer. The helper records the full replay command duration first, then speeds up the complete capture to fit the 45-second limit. The replay ZIP contains the same authoritative simulation record. Replays are version-specific, so open this one with **0 A.D. 0.28.x**.

If neither side reaches conquest within 1,000 RL turns, the notebook reports a turn-limit leader. The documented score weights population, active workers, military units, structures, and banked resources. The score is an experiment comparison metric rather than an official 0 A.D. victory condition.